# Quantum Idle Audit demonstrations

Six demonstrations, each the same circuit on the same IBM chip run two ways: the client's default settings, and calibrated with the gate error measured on the qubits the circuit uses.

**1. Replay** a recorded hardware result. No key, no IBM time. The finance replay below is already printed: select **Runtime > Run all** (Colab) or run the cells in order to recompute it here.
**2. Live run** on your own IBM Quantum account (optional), further down.

The code is the public repository [github.com/indahl-ai/demo-sandbox](https://github.com/indahl-ai/demo-sandbox) at release tag `v1.0`: the setup cell downloads that fixed commit, so a later change to the repository never runs here. On your own machine, in Colab or on a RunPod pod you start yourself, nothing is sent to Heuremen. On a sandbox hosted by QIA, the container is run by the operator and the private link opens a root session in it: use replay only there, or a key revoked afterwards. Heuremen is not affiliated with IBM.

In [ ]:
# Setup: about one minute the first time. Downloads the public repository and installs four libraries (qiskit, qiskit-ibm-runtime, numpy, scipy).
import os, sys, subprocess
if not os.path.exists('sandbox.py'):
    if os.path.exists('/workspace/demo/sandbox.py'): os.chdir('/workspace/demo')
    else:
        if not os.path.exists('demo-sandbox'): subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', 'v1.0', 'https://github.com/indahl-ai/demo-sandbox'], check=True)   # release tag, a fixed commit
        os.chdir('demo-sandbox')
PY = os.path.abspath('.venv/bin/python') if os.path.exists('.venv/bin/python') else sys.executable
if subprocess.run([PY, '-c', 'import qiskit, qiskit_ibm_runtime, numpy, scipy'], capture_output=True).returncode:   # install only what is missing
    if subprocess.run([PY, '-m', 'pip', '--version'], capture_output=True).returncode: subprocess.run([PY, '-m', 'ensurepip', '-q'], check=True)
    subprocess.run([PY, '-m', 'pip', 'install', '-q', '--disable-pip-version-check', '-r', 'requirements.txt'], check=True)
def sandbox(*args):
    p = subprocess.Popen([PY, 'sandbox.py', *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout: print(line, end='')
    p.wait()
print('Ready in', os.getcwd())

## Choose a demonstration
`finance`, `chemistry`, `optimization`, `simulation`, `error_correction` or `ai_kernels`. Chips: `kingston`, `fez`, `marrakesh`. Recorded runs: `1` or `2`.

In [ ]:
DEMO = 'finance'
CHIP = 'fez'
RUN = 1

## 1. Replay a recorded result (no key, no IBM time)
Re-analyzes the recorded hardware counts and prints the full output, then a RESULT box that restates the headline in plain words.

In [1]:
sandbox(DEMO, 'replay', '--chip', CHIP, '--run', str(RUN))

REPLAY of recorded run 1 on ibm_fez. Nothing is sent anywhere; no key is used.
Recorded hardware counts and benchmarking data; the circuits are rebuilt on a stand-in chip only to count depth and two-qubit gates.
pair: stated / RB-measured / ratio today: [((51, 58), 0.00414, 0.003, 0.72), ((58, 71), 0.0025, 0.00172, 0.69)]
frozen scale 1.35 (calibrated on the chip's EARLIER v2 job, applied unchanged to S and O; nothing fitted on these shots)
backend=ibm_fez exact a=0.197980 exact price=8.25433
exact price 8.2543
  N no fade (client): price 7.9974 (99% interval 7.4620 to 8.5329), mean abs error 1.0457; pooled-shots price 8.9468 error 0.6925
  S fade at stated CZ error: price 8.2044 (99% interval 8.1741 to 8.2347), mean abs error 0.0627; pooled-shots price 8.2058 error 0.0485
  O fade at our measured CZ error: price 8.2827 (99% interval 8.2455 to 8.3200), mean abs error 0.0674; pooled-shots price 8.2841 error 0.0298
  OUTCOME ours minus S (error difference, 99% interval -0.0263 to +0.0358

## 2. Live run on your own IBM Quantum account (optional)
Set `LIVE = True` in the next cell and run it. Enter the API key when asked. It is held in this session only: not shown, not printed, not saved. If the key has more than one IBM instance, `check` lists them with the time left on each: put the name in `INSTANCE` and run the cell again.

In [ ]:
LIVE = False          # change to True to use your own IBM Quantum account
INSTANCE = ''          # only if asked, for example 'my-instance'
if LIVE:
    from getpass import getpass
    if not os.environ.get('IBM_QUANTUM_TOKEN'): os.environ['IBM_QUANTUM_TOKEN'] = getpass('IBM Quantum API key: ')
    if INSTANCE: os.environ['DEMO_INSTANCE'] = INSTANCE
    sandbox(DEMO, 'check', '--chip', CHIP)   # remaining IBM time and the estimate. Submits nothing.
else:
    print('Live run is off. Set LIVE = True in this cell and run it again to use your own IBM Quantum account.')

### Fire
Submits the jobs on your IBM account and uses your IBM time (see the estimate above). Change `False` to `True` to confirm.

In [ ]:
USE_MY_IBM_TIME = False
if LIVE and USE_MY_IBM_TIME: sandbox(DEMO, 'fire', '--chip', CHIP, '--yes')
else: print('Not submitted. Set LIVE = True above and USE_MY_IBM_TIME = True here to submit.')

### Collect and compare
Checks the jobs every 30 seconds and prints the comparison when they finish (usually a few minutes, longer if the chip has a queue).

In [ ]:
import time
res = None
for i in range(60 if LIVE and USE_MY_IBM_TIME else 0):
    f = subprocess.run([PY, 'sandbox.py', DEMO, 'fetch', '--chip', CHIP], capture_output=True, text=True)
    out = f.stdout
    print(time.strftime('%H:%M:%S'), out.strip().splitlines()[-1] if out.strip() else '')
    if f.returncode and 'not finished' not in f.stdout + f.stderr:   # a real failure, not a job still running: show it and stop
        print(out[-3000:]); print(f.stderr[-3000:])
        raise RuntimeError('fetch failed (return code %d); stopped. The error is printed above.' % f.returncode)
    res = subprocess.run([PY, 'sandbox.py', DEMO, 'analyze', '--chip', CHIP], capture_output=True, text=True)
    if ' RESULT ' in res.stdout and 'No summary' not in res.stdout:
        print(res.stdout); break
    time.sleep(30)
else:
    print('Nothing to collect yet.' if not (LIVE and USE_MY_IBM_TIME) else 'Not finished after 30 minutes. Run this cell again later.')
    if res is not None and res.returncode: print('Last analyze error output:'); print(res.stderr[-3000:])

Delete the session when finished (Colab: **Runtime > Disconnect and delete runtime**). Everything in it, including your live run, is removed.